In [ ]:
!pip install datasets transformers torch pandas matplotlib spacy
!python -m spacy download en_core_web_sm

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
import re
from datasets import load_dataset
import spacy

In [ ]:

PHASE2_ANALYSIS = "phase2_analysis.json"  
OUTPUT_JSON = "phase3a_squad_analysis.json"
SAMPLE_SIZE = 50000  

squad = load_dataset("squad_v2")

train_data = squad['train']
val_data = squad['validation']

if SAMPLE_SIZE:
    train_sample = train_data.select(range(min(SAMPLE_SIZE, len(train_data))))
else:
    train_sample = train_data

In [ ]:
train_sample[0]

In [ ]:

# questionword count
question_types = {
    'what': 0,
    'who': 0,
    'when': 0,
    'where': 0,
    'why': 0,
    'how': 0,
    'which': 0,
    'whose': 0,
    'other': 0
}

question_examples = defaultdict(list)

for example in train_sample:
    question = example['question'].lower().strip()
    
    # question type
    found = False
    for qtype in ['what', 'who', 'when', 'where', 'why', 'how', 'which', 'whose']:
        if question.startswith(qtype):
            question_types[qtype] += 1
            
            if len(question_examples[qtype]) < 5:
                question_examples[qtype].append({
                    'question': example['question'],
                    'answer': example['answers']['text'][0] if example['answers']['text'] else '[no answer]'
                })
            
            found = True
            break
    
    if not found:
        question_types['other'] += 1


# results
total_questions = sum(question_types.values())

# sorted result
sorted_types = sorted(question_types.items(), key=lambda x: x[1], reverse=True)
print(sorted_types)

for qtype, count in sorted_types:
    percentage = (count / total_questions) * 100
    print(f"{qtype.upper():8s}: {count:6,d} ({percentage:5.1f}%)")


for qtype in ['what', 'who', 'when', 'where', 'how']:
    if question_examples[qtype]:
        print(f"\n{qtype.upper()} question:")
        for i, ex in enumerate(question_examples[qtype][:1],1):
            print(f"  {i}. Q: {ex['question']}")
            print(f"     A: {ex['answer']}")

In [ ]:

# answerability
answerable_count = 0
unanswerable_count = 0
answer_lengths = []

for example in train_sample:
    if example['answers']['text']:
        answerable_count += 1
        answer = example['answers']['text'][0]
        answer_lengths.append(len(answer.split()))
    else:
        unanswerable_count += 1


print(answerable_count)
print(unanswerable_count)


In [ ]:
#answer entity types
nlp = spacy.load("en_core_web_sm")

answer_entities = defaultdict(list)
sample_limit = min(50000, answerable_count)

processed = 0
for example in train_sample:
    if example['answers']['text']:
        answer = example['answers']['text'][0]
        doc = nlp(answer)
        
        for ent in doc.ents:
            answer_entities[ent.label_].append(ent.text)
        
        processed += 1
        if processed >= sample_limit:
            break


entity_counts = {label: len(entities) for label, entities in answer_entities.items()}
sorted_entities = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)

for label, count in sorted_entities[:15]:  
    percentage = (count / sum(entity_counts.values())) * 100
    print(f"{label:15s}: {count:5d} ({percentage:5.1f}%)")


In [ ]:

# question_answer mapping
question_answer_mapping = defaultdict(lambda: defaultdict(int))

sample_limit = min(50000, len(train_sample))

for i, example in enumerate(train_sample):
    if i >= sample_limit:
        break
    
    if not example['answers']['text']:
        continue
    
    question = example['question'].lower().strip()
    answer = example['answers']['text'][0]
    
    # qtype 
    qtype = 'other'
    for qt in ['what', 'who', 'when', 'where', 'why', 'how', 'which', 'whose']:
        if question.startswith(qt):
            qtype = qt
            break
    
    # answer entity
    doc = nlp(answer)
    if doc.ents:
        for ent in doc.ents:
            question_answer_mapping[qtype][ent.label_] += 1
    else:
        question_answer_mapping[qtype]['NO_ENTITY'] += 1


# results
for qtype in ['who', 'when', 'where', 'what']:
    if qtype in question_answer_mapping:
        print(f"\n{qtype.upper()} question's answer entity:")
        
        entities = question_answer_mapping[qtype]
        sorted_entities = sorted(entities.items(), key=lambda x: x[1], reverse=True)[:5]
        
        total = sum(entities.values())
        for entity_type, count in sorted_entities:
            percentage = (count / total) * 100
            print(f"  {entity_type:15s}: {count:4d} ({percentage:5.1f}%)")

In [ ]:

analysis_results = {
    'metadata': {
        'dataset': 'SQuAD v2',
        'total_train': len(train_data),
        'total_validation': len(val_data),
        'analyzed_samples': len(train_sample)
    },
    'question_types': {
        'distribution': dict(question_types),
        'examples': {k: v for k, v in question_examples.items()}
    },
    'answers': {
        'answerable': answerable_count,
        'unanswerable': unanswerable_count,
        'avg_length': float(np.mean(answer_lengths)) if answer_lengths else 0,
        'median_length': float(np.median(answer_lengths)) if answer_lengths else 0,
        'entity_types': dict(entity_counts)
    },
    'question_answer_mapping': {
        qtype: dict(entities) 
        for qtype, entities in question_answer_mapping.items()
    }
}

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)
